[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_2_3/01_tag_2_3_neuron_scratch_klassifikation.ipynb)

# Tag 2/3 - 01 Ein Neuron von Scratch

Ein einzelnes Neuron berechnet eine gewichtete Summe und schickt sie durch eine Aktivierungsfunktion:

`p = sigmoid(w1 * x1 + w2 * x2 + b)`

`p` ist die vorhergesagte Wahrscheinlichkeit für Klasse 1. Das Notebook baut dieses Neuron ohne TensorFlow, Keras oder scikit-learn. NumPy erzeugt nur die Daten und Matplotlib zeichnet die Grafiken.


In [ ]:
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

RANDOM_STATE = 42
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True
print("Setup abgeschlossen.")


## Datensatz mit einstellbarer Überlappung

Zwei Klassen liegen in einer 2D-Ebene. Mit dem Slider `Überlappung` wird das Problem schwieriger: kleine Werte trennen die Klassen klar, große Werte lassen die Klassen stark ineinanderlaufen. Das ist wichtig für Recall, weil bei überlappenden Klassen mehr positive Beispiele übersehen werden können.


In [ ]:
def make_data(n_per_class=80, overlap=0.65, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    class_0 = rng.normal(loc=(-1.1, -0.8), scale=overlap, size=(n_per_class, 2))
    class_1 = rng.normal(loc=(1.1, 0.8), scale=overlap, size=(n_per_class, 2))
    X_np = np.vstack([class_0, class_1]).astype(float)
    y_np = np.array([0] * n_per_class + [1] * n_per_class, dtype=int)
    order = rng.permutation(len(y_np))
    X_np = X_np[order]
    y_np = y_np[order]
    return X_np.tolist(), y_np.tolist(), X_np, y_np


def plot_dataset(overlap=0.65):
    X, y, X_np, y_np = make_data(overlap=overlap)
    plt.figure(figsize=(6, 5))
    plt.scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", edgecolor="black", alpha=0.85)
    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(f"Klassifikationsdaten mit Überlappung={overlap:.2f}")
    plt.show()
    print("Samples:", len(y))
    print("Klassenverteilung:", {klasse: y.count(klasse) for klasse in sorted(set(y))})


widgets.interact(
    plot_dataset,
    overlap=widgets.FloatSlider(value=0.65, min=0.25, max=1.60, step=0.05, description="Überlappung"),
);


## Modell, Loss und Metriken

Neben Accuracy berechnen wir auch Precision und Recall:

- **Precision**: Von den vorhergesagten positiven Fällen, wie viele waren wirklich positiv?
- **Recall**: Von den wirklich positiven Fällen, wie viele hat das Neuron gefunden?

Recall ist besonders wichtig, wenn False Negatives teuer sind, zum Beispiel bei medizinischen Screening-Problemen.


In [ ]:
def sigmoid(z):
    z = max(min(z, 60), -60)
    return 1.0 / (1.0 + math.exp(-z))


def predict_proba(sample, w1, w2, b):
    return sigmoid(w1 * sample[0] + w2 * sample[1] + b)


def predict_class(sample, w1, w2, b, threshold=0.5):
    return int(predict_proba(sample, w1, w2, b) >= threshold)


def binary_cross_entropy(target, proba):
    eps = 1e-12
    proba = min(max(proba, eps), 1.0 - eps)
    return -(target * math.log(proba) + (1 - target) * math.log(1.0 - proba))


def evaluate(X, y, w1, w2, b):
    losses = []
    tp = fp = tn = fn = 0
    for sample, target in zip(X, y):
        proba = predict_proba(sample, w1, w2, b)
        pred = int(proba >= 0.5)
        losses.append(binary_cross_entropy(target, proba))
        if pred == 1 and target == 1:
            tp += 1
        elif pred == 1 and target == 0:
            fp += 1
        elif pred == 0 and target == 0:
            tn += 1
        else:
            fn += 1

    accuracy = (tp + tn) / len(y)
    precision = tp / max(1, tp + fp)
    recall = tp / max(1, tp + fn)
    return {
        "loss": sum(losses) / len(losses),
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn,
    }


def train_neuron(X, y, learning_rate=0.35, epochs=220):
    w1, w2, b = 0.0, 0.0, 0.0
    history = []

    for epoch in range(epochs):
        grad_w1 = 0.0
        grad_w2 = 0.0
        grad_b = 0.0

        for sample, target in zip(X, y):
            proba = predict_proba(sample, w1, w2, b)

            # Fehlerterm für Sigmoid + Binary-Cross-Entropy.
            error = proba - target
            grad_w1 += error * sample[0]
            grad_w2 += error * sample[1]
            grad_b += error

        # Parameter im Fokus: learning_rate bestimmt die Schrittgröße.
        n = len(y)
        w1 -= learning_rate * grad_w1 / n
        w2 -= learning_rate * grad_w2 / n
        b -= learning_rate * grad_b / n

        metrics = evaluate(X, y, w1, w2, b)
        history.append({"epoch": epoch + 1, "w1": w1, "w2": w2, "b": b, **metrics})

    return (w1, w2, b), history


In [ ]:
def plot_confusion(metrics, ax):
    matrix = np.array([[metrics["tn"], metrics["fp"]], [metrics["fn"], metrics["tp"]]])
    ax.imshow(matrix, cmap="Blues")
    labels = [["TN", "FP"], ["FN", "TP"]]
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{labels[i][j]}\n{matrix[i, j]}", ha="center", va="center", fontsize=12)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["pred 0", "pred 1"])
    ax.set_yticklabels(["true 0", "true 1"])
    ax.set_title("Confusion Matrix")


def plot_neuron(X_np, y_np, X, y, w1, w2, b, title="Neuron"):
    xx, yy = np.meshgrid(
        np.linspace(X_np[:, 0].min() - 0.6, X_np[:, 0].max() + 0.6, 180),
        np.linspace(X_np[:, 1].min() - 0.6, X_np[:, 1].max() + 0.6, 180),
    )
    zz = 1.0 / (1.0 + np.exp(-(w1 * xx + w2 * yy + b)))
    metrics = evaluate(X, y, w1, w2, b)

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    axes[0].contourf(xx, yy, zz, levels=np.linspace(0, 1, 21), cmap="coolwarm", alpha=0.35)
    axes[0].contour(xx, yy, zz, levels=[0.5], colors="black", linewidths=2)
    axes[0].scatter(X_np[:, 0], X_np[:, 1], c=y_np, cmap="coolwarm", edgecolor="black", alpha=0.9)
    axes[0].set_xlabel("x1")
    axes[0].set_ylabel("x2")
    axes[0].set_title(title)
    plot_confusion(metrics, axes[1])
    plt.tight_layout()
    plt.show()

    print(f"Loss:      {metrics['loss']:.3f}")
    print(f"Accuracy:  {metrics['accuracy']:.1%}")
    print(f"Precision: {metrics['precision']:.1%}")
    print(f"Recall:    {metrics['recall']:.1%}")
    print(f"Gewichte:  w1={w1:.2f}, w2={w2:.2f}, b={b:.2f}")


## Manuell tunen

Ändere Gewichte, Bias und Datenüberlappung. Die schwarze Linie ist die Entscheidungsgrenze bei `p = 0.5`.


In [ ]:
def manual_demo(overlap=0.65, w1=1.0, w2=1.0, b=0.0):
    X, y, X_np, y_np = make_data(overlap=overlap)
    plot_neuron(X_np, y_np, X, y, w1, w2, b, title="Manuell eingestelltes Neuron")


widgets.interact(
    manual_demo,
    overlap=widgets.FloatSlider(value=0.65, min=0.25, max=1.60, step=0.05, description="Überlappung"),
    w1=widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description="w1"),
    w2=widgets.FloatSlider(value=1.0, min=-5.0, max=5.0, step=0.1, description="w2"),
    b=widgets.FloatSlider(value=0.0, min=-5.0, max=5.0, step=0.1, description="b"),
);


## Automatisch trainieren

Jetzt sucht Gradient Descent die Gewichte. Bei jedem Schritt wird der durchschnittliche Gradient über alle Trainingsdaten berechnet.

### Wie werden Gradient Descent und Backpropagation hier berechnet?

Für ein Sample mit zwei Merkmalen gilt:

1. **Forward Pass**

   Das Neuron berechnet zuerst die Voraktivierung und daraus eine Wahrscheinlichkeit:

   `z = w1 * x1 + w2 * x2 + b`

   `p = sigmoid(z) = 1 / (1 + exp(-z))`

2. **Loss**

   Für binäre Klassifikation nutzen wir Binary Cross Entropy:

   `loss = -(y * log(p) + (1 - y) * log(1 - p))`

   Wenn `y = 1` ist, wird ein kleines `p` stark bestraft. Wenn `y = 0` ist, wird ein großes `p` stark bestraft.

3. **Backpropagation**

   Backpropagation bedeutet: Wir leiten den Loss rückwärts nach den Parametern ab. Für Sigmoid plus Binary Cross Entropy vereinfacht sich der wichtigste Term zu:

   `dL/dz = p - y`

   Daraus folgen die Gradienten:

   `dL/dw1 = (p - y) * x1`

   `dL/dw2 = (p - y) * x2`

   `dL/db = p - y`

   Für viele Samples mitteln wir diese Werte über den ganzen Datensatz.

4. **Gradient Descent Update**

   Die Parameter werden in die Gegenrichtung des Gradienten verschoben:

   `w1 = w1 - learning_rate * dL/dw1`

   `w2 = w2 - learning_rate * dL/dw2`

   `b = b - learning_rate * dL/db`

   Die Lernrate bestimmt die Schrittgröße. Ist sie zu klein, lernt das Neuron langsam. Ist sie zu groß, kann der Loss springen oder sogar schlechter werden.


In [ ]:
def training_demo(overlap=0.65, learning_rate=0.35, epochs=220):
    X, y, X_np, y_np = make_data(overlap=overlap)
    trained_weights, history = train_neuron(X, y, learning_rate=learning_rate, epochs=epochs)
    w1, w2, b = trained_weights

    epochs_axis = [row["epoch"] for row in history]
    losses = [row["loss"] for row in history]
    recalls = [row["recall"] for row in history]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs_axis, losses)
    axes[0].set_title("Training Loss")
    axes[0].set_xlabel("Epoche")
    axes[0].set_ylabel("Binary Cross-Entropy")
    axes[1].plot(epochs_axis, recalls, color="green")
    axes[1].set_title("Training Recall")
    axes[1].set_xlabel("Epoche")
    axes[1].set_ylim(0.0, 1.02)
    plt.tight_layout()
    plt.show()

    plot_neuron(X_np, y_np, X, y, w1, w2, b, title="Trainiertes Neuron")


widgets.interact(
    training_demo,
    overlap=widgets.FloatSlider(value=0.65, min=0.25, max=1.60, step=0.05, description="Überlappung"),
    learning_rate=widgets.FloatSlider(value=0.35, min=0.01, max=1.20, step=0.01, description="Lernrate"),
    epochs=widgets.IntSlider(value=220, min=20, max=500, step=20, description="Epochen"),
);
